# 00 — Clean and merge PAL predictions (all models)

Merges the PAL bag-level prediction files into one clean file for every model × approach × history configuration.

- Models: Qwen, LLaVA, InternVL
- Approaches: `visual → appfr`, `sensor → appod`
- Histories: H02, H04, ..., H32
- Bags: 3
- Raw files: 288 (96 per model)
- Clean outputs: 96 (32 per model)
- Frame-level metrics: correctness, MSP, PCS, entropy, normalized entropy, and Deep Gini

The notebook verifies that all models, approaches, and histories use exactly the same PAL frames and ground-truth labels before writing outputs.


In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import display

ANALYSIS = Path.cwd().parent if Path.cwd().name == "Codes" else Path.cwd()
RAW = ANALYSIS / "Model_Pred" / "PAL" / "Pal_Pred"
CLEAN = ANALYSIS / "Outputs" / "PAL" / "clean"

MODELS = ["qwen", "llava", "internvl"]
HISTORIES = list(range(2, 33, 2))
MODE_TO_APPROACH = {"visual": "appfr", "sensor": "appod"}
EXPECTED_BAGS = 3
EXPECTED_FILES_PER_CONFIGURATION = 3
EXPECTED_CONFIGURATIONS = len(MODELS) * len(MODE_TO_APPROACH) * len(HISTORIES)
EXPECTED_RAW_FILES = EXPECTED_CONFIGURATIONS * EXPECTED_FILES_PER_CONFIGURATION

LABELS = ["safe", "potentially_unsafe", "unsafe"]
PROB_COLS = ["prob_safe", "prob_potentially_unsafe", "prob_unsafe"]
PROB_TO_LABEL = dict(zip(PROB_COLS, LABELS))

ALIASES = {
    "bag_name": ["bag_name"],
    "frame_id": ["frame_id"],
    "frame_time_s": ["frame_time_s"],
    "ground_truth": ["gt_state", "ground_truth", "ground_truth_label"],
    "model_saved_label": ["predicted_state", "predicted_label"],
    "prob_safe": ["p_safe", "prob_safe"],
    "prob_potentially_unsafe": ["p_potentially_unsafe", "prob_potentially_unsafe"],
    "prob_unsafe": ["p_unsafe", "prob_unsafe"],
    "prediction_valid": ["prediction_valid"],
    "input_mode": ["input_mode"],
    "history_frames": ["history_frames"],
    "model_name": ["model_name"],
}

FINAL_COLS = [
    "bag_name", "frame_id", "frame_time_s", "ground_truth",
    "model_saved_label", "raw_output_label", "probability_argmax_label",
    "predicted_label", "label_source", "prediction_corrected",
    "probability_label_disagreement",
    *PROB_COLS, "msp", "pcs", "entropy", "normalized_entropy",
    "deep_gini", "correct_binary",
]

print("Analysis folder:", ANALYSIS)
print("PAL raw folder:", RAW)
print("PAL clean folder:", CLEAN)
assert RAW.exists(), f"PAL raw folder does not exist: {RAW}"


In [ ]:
# Discover only prediction files inside the three model directories.
groups = {}
inventory_rows = []

for model in MODELS:
    model_total = 0
    assert (RAW / model).is_dir(), f"Missing model folder: {RAW / model}"

    for mode, approach in MODE_TO_APPROACH.items():
        mode_total = 0

        for history in HISTORIES:
            folder = RAW / model / mode / f"h{history:02d}"
            assert folder.is_dir(), f"Missing folder: {folder}"

            files = sorted(folder.glob("*.csv"))
            groups[(model, mode, history)] = files
            assert len(files) == EXPECTED_FILES_PER_CONFIGURATION, (
                f"{model} {mode} H{history:02d}: expected "
                f"{EXPECTED_FILES_PER_CONFIGURATION} bag files, found {len(files)}"
            )
            mode_total += len(files)

        inventory_rows.append({
            "model": model,
            "input_mode": mode,
            "approach": approach,
            "histories": len(HISTORIES),
            "files_per_history": EXPECTED_FILES_PER_CONFIGURATION,
            "raw_files": mode_total,
        })
        model_total += mode_total

    expected_per_model = len(MODE_TO_APPROACH) * len(HISTORIES) * EXPECTED_FILES_PER_CONFIGURATION
    assert model_total == expected_per_model, (
        f"{model}: expected {expected_per_model} raw files, found {model_total}"
    )

inventory = pd.DataFrame(inventory_rows)
assert len(groups) == EXPECTED_CONFIGURATIONS
assert sum(map(len, groups.values())) == EXPECTED_RAW_FILES
display(inventory)
print(f"Ready: {EXPECTED_RAW_FILES} PAL raw files in {EXPECTED_CONFIGURATIONS} configurations.")


In [ ]:
def clean_labels(series):
    return (
        series.astype(str).str.strip().str.lower().replace({
            "potentially unsafe": "potentially_unsafe",
            "potentially-unsafe": "potentially_unsafe",
            "potentiallyunsafe": "potentially_unsafe",
        })
    )


def valid_boolean(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    return (
        series.astype(str).str.strip().str.lower()
        .map({"true": True, "false": False, "1": True, "0": False})
        .fillna(False).astype(bool)
    )


def read_prediction(file):
    raw = pd.read_csv(file)
    rename = {}
    path_metadata = {
        "input_mode": file.parent.parent.name,
        "history_frames": int(file.parent.name.removeprefix("h")),
        "model_name": file.parent.parent.parent.name,
    }

    for target, choices in ALIASES.items():
        source = next((column for column in choices if column in raw.columns), None)
        if source is None:
            if target in path_metadata:
                continue
            raise ValueError(
                f"{file}: missing required field '{target}'. "
                f"Available columns: {list(raw.columns)}"
            )
        rename[source] = target
    selected = raw[list(rename)].rename(columns=rename)
    for target, value in path_metadata.items():
        if target not in selected.columns:
            selected[target] = value

    selected["model_raw_output"] = (
        raw["internvl_raw_output"]
        if "internvl_raw_output" in raw.columns
        else pd.Series(pd.NA, index=raw.index)
    )

    selected["source_file"] = file.name
    return selected


def add_frame_metrics(df):
    df = df.copy()
    df["ground_truth"] = clean_labels(df["ground_truth"])
    df["model_saved_label"] = clean_labels(df["model_saved_label"])
    df[PROB_COLS] = df[PROB_COLS].apply(pd.to_numeric, errors="coerce")

    probabilities = df[PROB_COLS].to_numpy(dtype=float)
    df["probability_argmax_label"] = (
        df[PROB_COLS].idxmax(axis=1).map(PROB_TO_LABEL)
    )

    # RQ1 uses the explicit classification label. InternVL's raw response
    # is authoritative because some repaired predicted_state values and
    # some self-reported probabilities disagree with that response.
    df["raw_output_label"] = df["model_saved_label"]
    internvl_mask = df["model_name"].astype(str).str.lower().eq("internvl")
    if internvl_mask.any():
        extracted = (
            df.loc[internvl_mask, "model_raw_output"]
            .astype(str)
            .str.extract(
                r'"?label"?\s*:\s*"?'
                r'(safe|potentially_unsafe|unsafe)',
                flags=re.IGNORECASE,
                expand=False,
            )
            .str.lower()
        )
        if extracted.isna().any():
            bad = df.loc[internvl_mask].loc[extracted[extracted.isna()].index, "source_file"].unique()
            raise ValueError(f"Could not extract InternVL raw label from files: {list(bad)}")
        df.loc[internvl_mask, "raw_output_label"] = extracted

    df["predicted_label"] = df["raw_output_label"]
    df["label_source"] = np.where(internvl_mask, "raw_output", "saved_prediction")
    df["prediction_corrected"] = df["model_saved_label"] != df["predicted_label"]
    df["probability_label_disagreement"] = (
        df["predicted_label"] != df["probability_argmax_label"]
    )
    df["correct_binary"] = (df["ground_truth"] == df["predicted_label"]).astype(int)
    df["msp"] = probabilities.max(axis=1)
    sorted_probabilities = np.sort(probabilities, axis=1)
    df["pcs"] = sorted_probabilities[:, -1] - sorted_probabilities[:, -2]
    df["entropy"] = -(probabilities * np.log(probabilities + 1e-12)).sum(axis=1)
    df["normalized_entropy"] = df["entropy"] / np.log(len(LABELS))
    df["deep_gini"] = 1 - np.square(probabilities).sum(axis=1)
    return df


In [ ]:
# Inspect one H02 bag for every model and input mode.
sample_rows = []

for model in MODELS:
    for mode in MODE_TO_APPROACH:
        sample_file = groups[(model, mode, 2)][0]
        sample = add_frame_metrics(read_prediction(sample_file))
        sample_rows.append({
            "model": model,
            "input_mode": mode,
            "file": sample_file.name,
            "rows": len(sample),
            "bags": sample["bag_name"].nunique(),
            "ground_truth_labels": sorted(sample["ground_truth"].unique()),
            "saved_prediction_labels": sorted(sample["model_saved_label"].unique()),
            "argmax_prediction_labels": sorted(sample["predicted_label"].unique()),
            "probability_sum_min": sample[PROB_COLS].sum(axis=1).min(),
            "probability_sum_max": sample[PROB_COLS].sum(axis=1).max(),
            "label_corrections": int(sample["prediction_corrected"].sum()),
        })

sample_check = pd.DataFrame(sample_rows)
display(sample_check)


In [ ]:
# Qwen AppFr H02 defines the canonical PAL frame set.
canonical_files = groups[("qwen", "visual", 2)]
canonical = pd.concat([read_prediction(file) for file in canonical_files], ignore_index=True)
canonical["ground_truth"] = clean_labels(canonical["ground_truth"])
canonical["frame_time_s"] = pd.to_numeric(canonical["frame_time_s"], errors="coerce")

assert canonical["bag_name"].nunique() == EXPECTED_BAGS
assert not canonical.duplicated(["bag_name", "frame_id"]).any()
assert set(canonical["ground_truth"]) <= set(LABELS)
assert np.isfinite(canonical["frame_time_s"]).all()

CANONICAL_ROWS = len(canonical)
canonical_identity = (
    canonical[["bag_name", "frame_id", "frame_time_s", "ground_truth"]]
    .sort_values(["bag_name", "frame_id"])
    .reset_index(drop=True)
)
canonical_bags = set(canonical["bag_name"].unique())

gt_by_bag = pd.crosstab(canonical["bag_name"], canonical["ground_truth"]).reindex(
    columns=LABELS, fill_value=0
)
gt_by_bag.loc["TOTAL"] = gt_by_bag.sum()
display(gt_by_bag)
print("Canonical PAL frames:", CANONICAL_ROWS)
print("Canonical PAL bags:", len(canonical_bags))


In [ ]:
validation_rows = []

for model in MODELS:
    for mode, approach in MODE_TO_APPROACH.items():
        for history in HISTORIES:
            files = groups[(model, mode, history)]
            df = pd.concat([read_prediction(file) for file in files], ignore_index=True)
            df = add_frame_metrics(df)
            name = f"{model} {approach} H{history:02d}"

            assert len(files) == EXPECTED_FILES_PER_CONFIGURATION
            assert df["bag_name"].nunique() == EXPECTED_BAGS, f"{name}: wrong bag count"
            assert set(df["bag_name"].unique()) == canonical_bags, f"{name}: wrong bags"
            assert len(df) == CANONICAL_ROWS, f"{name}: expected {CANONICAL_ROWS} rows, found {len(df)}"
            assert not df.duplicated(["bag_name", "frame_id"]).any(), f"{name}: duplicate frames"
            assert df["input_mode"].astype(str).str.strip().str.lower().eq(mode).all(), f"{name}: incorrect input_mode"
            assert pd.to_numeric(df["history_frames"], errors="coerce").eq(history).all(), f"{name}: incorrect history_frames"
            assert df["model_name"].astype(str).str.strip().str.lower().eq(model).all(), f"{name}: incorrect model_name"
            assert valid_boolean(df["prediction_valid"]).all(), f"{name}: invalid predictions present"
            assert set(df["ground_truth"]) <= set(LABELS), f"{name}: unknown ground-truth label"
            assert set(df["model_saved_label"]) <= set(LABELS), f"{name}: unknown saved prediction label"
            assert set(df["raw_output_label"]) <= set(LABELS), f"{name}: unknown raw-output label"
            assert set(df["probability_argmax_label"]) <= set(LABELS), f"{name}: unknown probability-argmax label"
            assert set(df["predicted_label"]) <= set(LABELS), f"{name}: unknown final prediction label"

            probabilities = df[PROB_COLS].to_numpy(dtype=float)
            assert np.isfinite(probabilities).all(), f"{name}: invalid probability"
            assert ((probabilities >= 0) & (probabilities <= 1)).all(), f"{name}: probability outside [0,1]"
            assert np.isclose(probabilities.sum(axis=1), 1.0, atol=1e-5).all(), f"{name}: probabilities do not sum to 1"
            assert not df[FINAL_COLS].isna().any().any(), f"{name}: missing required clean values"

            identity = (
                df[["bag_name", "frame_id", "frame_time_s", "ground_truth"]]
                .assign(frame_time_s=lambda x: pd.to_numeric(x["frame_time_s"], errors="coerce"))
                .sort_values(["bag_name", "frame_id"])
                .reset_index(drop=True)
            )
            assert identity[["bag_name", "frame_id", "ground_truth"]].equals(
                canonical_identity[["bag_name", "frame_id", "ground_truth"]]
            ), f"{name}: frame/ground-truth mismatch"
            assert np.allclose(
                identity["frame_time_s"], canonical_identity["frame_time_s"], atol=1e-6, rtol=0
            ), f"{name}: frame-time mismatch"

            validation_rows.append({
                "model": model, "approach": approach, "history": f"H{history:02d}",
                "source_files": len(files), "bags": df["bag_name"].nunique(),
                "rows": len(df),
                "label_corrections": int(df["prediction_corrected"].sum()),
                "probability_label_disagreements": int(df["probability_label_disagreement"].sum()),
                "accuracy_check": df["correct_binary"].mean(),
            })

validation = pd.DataFrame(validation_rows)
assert len(validation) == EXPECTED_CONFIGURATIONS
display(validation)
print(f"Validated: {len(validation)} PAL configurations.")


In [ ]:
CLEAN.mkdir(parents=True, exist_ok=True)
summary_rows, correction_rows = [], []

for model in MODELS:
    model_output = CLEAN / model
    model_output.mkdir(parents=True, exist_ok=True)

    for mode, approach in MODE_TO_APPROACH.items():
        for history in HISTORIES:
            files = groups[(model, mode, history)]
            df = pd.concat([read_prediction(file) for file in files], ignore_index=True)
            df = add_frame_metrics(df).sort_values(["bag_name", "frame_id"]).reset_index(drop=True)
            output_file = model_output / f"{model}_{approach}_h{history:02d}.csv"
            df[FINAL_COLS].to_csv(output_file, index=False)

            changed = df.loc[df["prediction_corrected"], [
                "bag_name", "frame_id", "ground_truth", "model_saved_label",
                "raw_output_label", "probability_argmax_label", "predicted_label",
                "label_source", *PROB_COLS,
            ]].copy()
            if not changed.empty:
                changed.insert(0, "history", f"H{history:02d}")
                changed.insert(0, "approach", approach)
                changed.insert(0, "model", model)
                correction_rows.append(changed)

            counts = df["predicted_label"].value_counts()
            summary_rows.append({
                "model": model, "approach": approach, "history": f"H{history:02d}",
                "history_frames": history, "source_files": len(files),
                "bags": df["bag_name"].nunique(), "rows": len(df),
                "corrected_saved_labels": int(df["prediction_corrected"].sum()),
                "probability_label_disagreements": int(df["probability_label_disagreement"].sum()),
                "accuracy_check": df["correct_binary"].mean(),
                "pred_safe": int(counts.get("safe", 0)),
                "pred_potentially_unsafe": int(counts.get("potentially_unsafe", 0)),
                "pred_unsafe": int(counts.get("unsafe", 0)),
                "output_file": output_file.name,
            })
            print(f"Saved {model} {approach} H{history:02d}")

summary = pd.DataFrame(summary_rows)
correction_columns = [
    "model", "approach", "history", "bag_name", "frame_id", "ground_truth",
    "model_saved_label", "raw_output_label", "probability_argmax_label",
    "predicted_label", "label_source", *PROB_COLS,
]
corrections = (
    pd.concat(correction_rows, ignore_index=True)
    if correction_rows else pd.DataFrame(columns=correction_columns)
)
summary.to_csv(CLEAN / "merge_summary.csv", index=False)
corrections.to_csv(CLEAN / "label_corrections.csv", index=False)
summary[[
    "model", "approach", "history", "rows",
    "corrected_saved_labels", "probability_label_disagreements",
]].to_csv(CLEAN / "label_probability_consistency_summary.csv", index=False)
gt_by_bag.to_csv(CLEAN / "pal_ground_truth_counts.csv")

clean_summary = summary.groupby("model").agg(
    clean_files=("output_file", "count"),
    rows_per_file=("rows", "first"),
    source_files=("source_files", "sum"),
    label_corrections=("corrected_saved_labels", "sum"),
)
assert clean_summary["clean_files"].eq(32).all()
assert clean_summary["rows_per_file"].eq(CANONICAL_ROWS).all()
assert clean_summary["source_files"].eq(96).all()
display(clean_summary)
print(f"\nFinished: {len(summary)} clean files saved in {CLEAN}")


In [ ]:
# Reload every output and verify the bytes written to disk.
expected_files = [
    CLEAN / model / f"{model}_{approach}_h{history:02d}.csv"
    for model in MODELS
    for approach in MODE_TO_APPROACH.values()
    for history in HISTORIES
]
missing_files = [str(file) for file in expected_files if not file.exists()]
assert not missing_files, f"Missing clean files: {missing_files}"

reload_rows = []
for file in expected_files:
    df = pd.read_csv(file)
    assert len(df) == CANONICAL_ROWS, f"{file.name}: found {len(df)} rows"
    assert df["bag_name"].nunique() == EXPECTED_BAGS, f"{file.name}: incorrect bag count"
    assert not df.duplicated(["bag_name", "frame_id"]).any(), f"{file.name}: duplicate frames"
    assert list(df.columns) == FINAL_COLS, f"{file.name}: unexpected columns"
    assert not df[FINAL_COLS].isna().any().any(), f"{file.name}: missing values"
    assert np.isclose(df[PROB_COLS].sum(axis=1), 1.0, atol=1e-5).all(), (
        f"{file.name}: invalid probability sums"
    )
    reload_rows.append({
        "file": file.name, "rows": len(df), "bags": df["bag_name"].nunique(),
        "accuracy": df["correct_binary"].mean(),
    })

reload_check = pd.DataFrame(reload_rows)
assert len(reload_check) == EXPECTED_CONFIGURATIONS
display(reload_check.groupby(reload_check["file"].str.split("_").str[0]).agg(
    clean_files=("file", "count"), rows_per_file=("rows", "first"), bags=("bags", "first")
))
print(f"Final verification passed: {len(reload_check)} PAL clean files.")
